# 41 – Parcels + Assessor Within TOCs / TODs

This notebook identifies **which parcels fall inside each TOC / TOD** and attaches assessor attributes.

**Goals:**
- Load cleaned parcels+assessor, TOC, and TOD layers.
- Ensure all layers share the same CRS.
- Spatially join parcels to TOCs/TODs.
- Save parcel-level files with TOC/TOD IDs attached (for later aggregation).

In [3]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

PARCELS_DIR = DATA_DIR / "parcels" / "processed"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "processed"

parcels_assessor_path = PARCELS_DIR / "maricopa_parcels_with_assessor.gpkg"
toc_clean_path = BOUNDARIES_DIR / "toc_clean.gpkg"


parcels_assessor_path, toc_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/maricopa_parcels_with_assessor.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/toc_clean.gpkg'))

In [5]:
parcels = gpd.read_file(parcels_assessor_path)
toc = gpd.read_file(toc_clean_path)

print("Parcels CRS:", parcels.crs)
print("TOC CRS:", toc.crs)

Parcels CRS: EPSG:2868
TOC CRS: EPSG:2868


In [8]:
# these are both already using the project CRS natively, but just in case:

PROJECT_CRS = "EPSG:2868"

if toc.crs != PROJECT_CRS:
    toc = toc.to_crs(PROJECT_CRS)
    print("Reprojected TOC to project CRS:", PROJECT_CRS)
if parcels.crs != PROJECT_CRS:
    parcels = parcels.to_crs(PROJECT_CRS)
    print("Reprojected parcels to project CRS:", PROJECT_CRS)

print("TOC CRS:", toc.crs)
print("Parcels CRS:", parcels.crs)

TOC CRS: EPSG:2868
Parcels CRS: EPSG:2868


In [9]:
toc.head()

,OBJECTID,APPLICABIL,geometry
0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89..."
1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89..."
2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90..."
3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915..."
4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91..."


In [ ]:
# Join parcels to TOCs (inner or left join depending on what you want)

parcels_in_toc = gpd.sjoin(
    parcels,
    toc[["APPLICABIL", "geometry"]],
    how="left",
    predicate="intersects"
)

parcels_in_toc.head()

,APN,Floor,LandSize,StartDate,Shp_Area,Shp_Length,PARCEL_NO,PROPERTYUSECODE,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS,NEIGHBORHOODCODE,MARKETAREACODE,geometry,index_right,APPLICABIL
0,10101001C,1,7565.0,2008-09-22,7505.147490,350.125679,10101001C,5400.0,68700.0,68700.0,0.0,68700.0,11336.0,7565.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581182.478 886654.38 0, 5811...",NaN,NaN
1,10101010,1,195864.0,2008-09-22,195989.095940,1838.766520,10101010,9780.0,8667760.0,1196500.0,7471260.0,7435591.0,1115339.0,195864.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((582322.095 889960.815 0, 582...",NaN,NaN
2,10101011,1,95491.0,2008-09-22,95501.799733,1248.957937,10101011,9720.0,5190674.0,636700.0,4553974.0,3377292.0,506594.0,95491.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581500.432 889688.576 0, 581...",NaN,NaN
3,10101012,1,66960.0,2008-09-22,66976.092596,1014.388578,10101012,9705.0,553400.0,553400.0,0.0,305062.0,45759.0,66960.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581428.706 889618.792 0, 581...",NaN,NaN
4,10101014,1,126838.0,2008-09-22,126838.044773,1424.568514,10101014,9720.0,7074148.0,969800.0,6104348.0,2971142.0,445671.0,126838.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581359.726 889391.699 0, 581...",NaN,NaN


In [21]:
# verifying join results

# parcels_in_toc['APPLICABIL'].isna().value_counts() (False means matched to a TOC and/or station area)

# parcels_in_toc[parcels_in_toc['APPLICABIL'].notna()].head() show head() for matched parcels only

parcels_in_toc.groupby('APPLICABIL').size() # count of parcels per TOC and/or station area

APPLICABIL
50th Street Station Area              270
Capitol Extension Area               2396
I-10 West Extension Area            14840
Northwest Extension Area II          2484
TOD District - 19North               5306
TOD District - Eastlake Garfield     3323
TOD District - Gateway               3259
TOD District - Midtown               4552
TOD District - Solano                3414
TOD District - South Central         5800
TOD District - Uptown                3746
dtype: int64

In [22]:
parcels_in_toc_path = PARCELS_DIR / "parcels_assessor_toc.gpkg"
parcels_in_toc.to_file(parcels_in_toc_path, driver="GPKG")

print("Saved parcels + assessor + TOC/TOD:", parcels_in_toc_path)

Saved parcels + assessor + TOC/TOD: c:\Users\arjav\DevSource\toc-performance-dashboard\data\parcels\processed\parcels_assessor_toc.gpkg
